In [2]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [3]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

## 1. Configuração

Por padrão, o gerador procura os notebooks na mesma pasta deste arquivo ou na pasta anterior.  
Ajuste apenas os caminhos se a estrutura do repositório mudar.

In [4]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [5]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


## _______________________________________________________________ "" ____________________________________________________________

In [6]:
# ============================================================
# WEEKEND MESSAGE OPTIMIZATION
# 01 — UNDERSTAND ENGAGEMENT VARIABLES
# ============================================================

import pandas as pd
import numpy as np

df = wa.copy()
df["sent_at"] = pd.to_datetime(df["sent_at"])

# ------------------------------------------------------------
# 1. Inspect engagement fields
# ------------------------------------------------------------

print("=" * 80)
print("DELIVERY STATUS")
print("=" * 80)

display(
    df["delivery_status"]
    .value_counts(dropna=False)
    .rename("messages")
    .to_frame()
)

print("\n" + "=" * 80)
print("INTERACTION")
print("=" * 80)

display(
    df["interaction"]
    .value_counts(dropna=False)
    .rename("messages")
    .to_frame()
)

# ------------------------------------------------------------
# 2. Cross delivery_status × interaction
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DELIVERY STATUS × INTERACTION")
print("=" * 80)

display(
    pd.crosstab(
        df["delivery_status"],
        df["interaction"],
        margins=True,
        dropna=False
    )
)

DELIVERY STATUS


,messages
delivery_status,
delivered,63543
failed_blocked,5051
failed_invalid_number,4723
failed_unreachable,2089



INTERACTION


,messages
interaction,
none,45201
read,17505
clicked_link,8145
replied,4555



DELIVERY STATUS × INTERACTION


interaction,clicked_link,none,read,replied,All
delivery_status,,,,,
delivered,8145,33338,17505,4555,63543
failed_blocked,0,5051,0,0,5051
failed_invalid_number,0,4723,0,0,4723
failed_unreachable,0,2089,0,0,2089
All,8145,45201,17505,4555,75406


1. Vamos provar primeiro o efeito puro de weekday/weekend no engagement : falhas não podem virar none e penalizar artificialmente read/click de um dia.

In [7]:
# ============================================================
# WEEKEND MESSAGE OPTIMIZATION
# 02 — ENGAGEMENT: WEEKDAY VS WEEKEND
# Base = DELIVERED messages only
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

eng = wa.copy()
eng["sent_at"] = pd.to_datetime(eng["sent_at"])

# ------------------------------------------------------------
# 1. Keep only successfully delivered messages
# ------------------------------------------------------------

eng = eng.loc[
    eng["delivery_status"].eq("delivered")
].copy()

# ------------------------------------------------------------
# 2. Calendar variables
# ------------------------------------------------------------

eng["weekday_num"] = eng["sent_at"].dt.dayofweek

eng["day_of_week"] = eng["sent_at"].dt.day_name()

eng["send_day_type"] = np.where(
    eng["weekday_num"] >= 5,
    "Weekend",
    "Weekday"
)

# ------------------------------------------------------------
# 3. Engagement flags
# ------------------------------------------------------------

eng["read_flag"] = eng["interaction"].eq("read")
eng["click_flag"] = eng["interaction"].eq("clicked_link")
eng["reply_flag"] = eng["interaction"].eq("replied")
eng["none_flag"] = eng["interaction"].eq("none")

eng["any_engagement_flag"] = eng["interaction"].isin(
    ["read", "clicked_link", "replied"]
)

# ------------------------------------------------------------
# 4. Weekday vs Weekend summary
# ------------------------------------------------------------

summary = (
    eng.groupby("send_day_type", observed=False)
       .agg(
           delivered_messages=("message_id", "nunique"),
           customers=("customer_id", "nunique"),

           none=("none_flag", "sum"),
           reads=("read_flag", "sum"),
           clicks=("click_flag", "sum"),
           replies=("reply_flag", "sum"),
           any_engagement=("any_engagement_flag", "sum")
       )
       .reset_index()
)

for metric in ["none", "reads", "clicks", "replies", "any_engagement"]:
    summary[f"{metric}_rate"] = (
        summary[metric] /
        summary["delivered_messages"]
    )

print("=" * 100)
print("ENGAGEMENT — WEEKDAY VS WEEKEND")
print("Base: successfully delivered messages")
print("=" * 100)

display(summary)

ENGAGEMENT — WEEKDAY VS WEEKEND
Base: successfully delivered messages


,send_day_type,delivered_messages,customers,none,reads,clicks,replies,any_engagement,none_rate,reads_rate,clicks_rate,replies_rate,any_engagement_rate
0,Weekday,54620,10845,27931,15449,7233,4007,26689,0.51,0.28,0.13,0.07,0.49
1,Weekend,8923,5986,5407,2056,912,548,3516,0.61,0.23,0.10,0.06,0.39


In [8]:
# ============================================================
# WEEKEND MESSAGE OPTIMIZATION
# 02 — ENGAGEMENT: WEEKDAY VS WEEKEND
# Base = DELIVERED messages only
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

eng = wa.copy()
eng["sent_at"] = pd.to_datetime(eng["sent_at"])

# ------------------------------------------------------------
# 1. Keep only successfully delivered messages
# ------------------------------------------------------------

eng = eng.loc[
    eng["delivery_status"].eq("delivered")
].copy()

# ------------------------------------------------------------
# 2. Calendar variables
# ------------------------------------------------------------

eng["weekday_num"] = eng["sent_at"].dt.dayofweek

eng["day_of_week"] = eng["sent_at"].dt.day_name()

eng["send_day_type"] = np.where(
    eng["weekday_num"] >= 5,
    "Weekend",
    "Weekday"
)

# ------------------------------------------------------------
# 3. Engagement flags
# ------------------------------------------------------------

eng["read_flag"] = eng["interaction"].eq("read")
eng["click_flag"] = eng["interaction"].eq("clicked_link")
eng["reply_flag"] = eng["interaction"].eq("replied")
eng["none_flag"] = eng["interaction"].eq("none")

eng["any_engagement_flag"] = eng["interaction"].isin(
    ["read", "clicked_link", "replied"]
)

# ------------------------------------------------------------
# 4. Weekday vs Weekend summary
# ------------------------------------------------------------

summary = (
    eng.groupby("send_day_type", observed=False)
       .agg(
           delivered_messages=("message_id", "nunique"),
           customers=("customer_id", "nunique"),

           none=("none_flag", "sum"),
           reads=("read_flag", "sum"),
           clicks=("click_flag", "sum"),
           replies=("reply_flag", "sum"),
           any_engagement=("any_engagement_flag", "sum")
       )
       .reset_index()
)

for metric in ["none", "reads", "clicks", "replies", "any_engagement"]:
    summary[f"{metric}_rate"] = (
        summary[metric] /
        summary["delivered_messages"]
    )

print("=" * 100)
print("ENGAGEMENT — WEEKDAY VS WEEKEND")
print("Base: successfully delivered messages")
print("=" * 100)

display(summary)

ENGAGEMENT — WEEKDAY VS WEEKEND
Base: successfully delivered messages


,send_day_type,delivered_messages,customers,none,reads,clicks,replies,any_engagement,none_rate,reads_rate,clicks_rate,replies_rate,any_engagement_rate
0,Weekday,54620,10845,27931,15449,7233,4007,26689,0.51,0.28,0.13,0.07,0.49
1,Weekend,8923,5986,5407,2056,912,548,3516,0.61,0.23,0.10,0.06,0.39


In [9]:
# ============================================================
# ENGAGEMENT BY DAY OF WEEK
# ============================================================

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

daily_engagement = (
    eng.groupby("day_of_week", observed=False)
       .agg(
           messages=("message_id", "nunique"),
           customers=("customer_id", "nunique"),
           reads=("read_flag", "sum"),
           clicks=("click_flag", "sum"),
           replies=("reply_flag", "sum"),
           any_engagement=("any_engagement_flag", "sum"),
           no_engagement=("none_flag", "sum")
       )
       .reindex(day_order)
       .reset_index()
)

for metric in [
    "reads",
    "clicks",
    "replies",
    "any_engagement",
    "no_engagement"
]:
    daily_engagement[f"{metric}_rate"] = (
        daily_engagement[metric] /
        daily_engagement["messages"]
    )

display(
    daily_engagement[
        [
            "day_of_week",
            "messages",
            "customers",
            "reads_rate",
            "clicks_rate",
            "replies_rate",
            "any_engagement_rate",
            "no_engagement_rate"
        ]
    ].style.format({
        "reads_rate": "{:.2%}",
        "clicks_rate": "{:.2%}",
        "replies_rate": "{:.2%}",
        "any_engagement_rate": "{:.2%}",
        "no_engagement_rate": "{:.2%}"
    })
)

,day_of_week,messages,customers,reads_rate,clicks_rate,replies_rate,any_engagement_rate,no_engagement_rate
0,Monday,11683,7385,27.95%,13.53%,7.48%,48.96%,51.04%
1,Tuesday,10776,6862,29.10%,12.96%,7.74%,49.81%,50.19%
2,Wednesday,10671,6809,29.05%,13.89%,7.46%,50.40%,49.60%
3,Thursday,10813,6919,27.86%,13.34%,6.96%,48.15%,51.85%
4,Friday,10677,6852,27.50%,12.47%,7.02%,46.99%,53.01%
5,Saturday,5531,4403,24.43%,10.90%,6.83%,42.16%,57.84%
6,Sunday,3392,2941,20.78%,9.11%,5.01%,34.91%,65.09%


In [10]:
# ============================================================
# WEEKEND OPTIMIZATION
# 03 — DAY OF WEEK × TEMPLATE
# Base = delivered messages
# ============================================================

template_day = (
    eng.groupby(
        ["template", "day_of_week"],
        observed=False
    )
    .agg(
        messages=("message_id", "nunique"),
        customers=("customer_id", "nunique"),
        reads=("read_flag", "sum"),
        clicks=("click_flag", "sum"),
        replies=("reply_flag", "sum"),
        any_engagement=("any_engagement_flag", "sum"),
        no_engagement=("none_flag", "sum")
    )
    .reset_index()
)

for metric in [
    "reads",
    "clicks",
    "replies",
    "any_engagement",
    "no_engagement"
]:
    template_day[f"{metric}_rate"] = (
        template_day[metric] /
        template_day["messages"]
    )

# Day ordering
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

template_day["day_of_week"] = pd.Categorical(
    template_day["day_of_week"],
    categories=day_order,
    ordered=True
)

template_day = template_day.sort_values(
    ["template", "day_of_week"]
)

display(
    template_day[
        [
            "template",
            "day_of_week",
            "messages",
            "customers",
            "reads_rate",
            "clicks_rate",
            "replies_rate",
            "any_engagement_rate",
            "no_engagement_rate"
        ]
    ].style.format({
        "reads_rate": "{:.2%}",
        "clicks_rate": "{:.2%}",
        "replies_rate": "{:.2%}",
        "any_engagement_rate": "{:.2%}",
        "no_engagement_rate": "{:.2%}"
    })
)

,template,day_of_week,messages,customers,reads_rate,clicks_rate,replies_rate,any_engagement_rate,no_engagement_rate
1,discount_offer,Monday,859,785,20.26%,16.07%,6.87%,43.19%,56.81%
5,discount_offer,Tuesday,755,704,22.78%,18.41%,6.89%,48.08%,51.92%
6,discount_offer,Wednesday,768,715,22.92%,17.58%,6.51%,47.01%,52.99%
4,discount_offer,Thursday,800,744,23.12%,20.25%,5.88%,49.25%,50.75%
0,discount_offer,Friday,793,746,19.55%,16.65%,6.56%,42.75%,57.25%
2,discount_offer,Saturday,417,402,19.18%,15.35%,6.47%,41.01%,58.99%
3,discount_offer,Sunday,277,271,15.16%,11.91%,3.61%,30.69%,69.31%
8,friendly_reminder,Monday,4032,3665,35.66%,9.03%,7.17%,51.86%,48.14%
12,friendly_reminder,Tuesday,3696,3380,35.80%,8.63%,7.60%,52.03%,47.97%
13,friendly_reminder,Wednesday,3592,3288,36.61%,8.24%,7.74%,52.59%,47.41%


In [11]:
# ============================================================
# WEEKEND OPTIMIZATION
# 03 — DAY OF WEEK × TEMPLATE
# Base = delivered messages
# ============================================================

template_day = (
    eng.groupby(
        ["template", "day_of_week"],
        observed=False
    )
    .agg(
        messages=("message_id", "nunique"),
        customers=("customer_id", "nunique"),
        reads=("read_flag", "sum"),
        clicks=("click_flag", "sum"),
        replies=("reply_flag", "sum"),
        any_engagement=("any_engagement_flag", "sum"),
        no_engagement=("none_flag", "sum")
    )
    .reset_index()
)

for metric in [
    "reads",
    "clicks",
    "replies",
    "any_engagement",
    "no_engagement"
]:
    template_day[f"{metric}_rate"] = (
        template_day[metric] /
        template_day["messages"]
    )

# Day ordering
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

template_day["day_of_week"] = pd.Categorical(
    template_day["day_of_week"],
    categories=day_order,
    ordered=True
)

template_day = template_day.sort_values(
    ["template", "day_of_week"]
)

display(
    template_day[
        [
            "template",
            "day_of_week",
            "messages",
            "customers",
            "reads_rate",
            "clicks_rate",
            "replies_rate",
            "any_engagement_rate",
            "no_engagement_rate"
        ]
    ].style.format({
        "reads_rate": "{:.2%}",
        "clicks_rate": "{:.2%}",
        "replies_rate": "{:.2%}",
        "any_engagement_rate": "{:.2%}",
        "no_engagement_rate": "{:.2%}"
    })
)

,template,day_of_week,messages,customers,reads_rate,clicks_rate,replies_rate,any_engagement_rate,no_engagement_rate
1,discount_offer,Monday,859,785,20.26%,16.07%,6.87%,43.19%,56.81%
5,discount_offer,Tuesday,755,704,22.78%,18.41%,6.89%,48.08%,51.92%
6,discount_offer,Wednesday,768,715,22.92%,17.58%,6.51%,47.01%,52.99%
4,discount_offer,Thursday,800,744,23.12%,20.25%,5.88%,49.25%,50.75%
0,discount_offer,Friday,793,746,19.55%,16.65%,6.56%,42.75%,57.25%
2,discount_offer,Saturday,417,402,19.18%,15.35%,6.47%,41.01%,58.99%
3,discount_offer,Sunday,277,271,15.16%,11.91%,3.61%,30.69%,69.31%
8,friendly_reminder,Monday,4032,3665,35.66%,9.03%,7.17%,51.86%,48.14%
12,friendly_reminder,Tuesday,3696,3380,35.80%,8.63%,7.60%,52.03%,47.97%
13,friendly_reminder,Wednesday,3592,3288,36.61%,8.24%,7.74%,52.59%,47.41%


DPD × day type × template

In [12]:
# ============================================================
# WEEKEND OPTIMIZATION
# 04 — TEMPLATE × DPD × WEEKDAY/WEEKEND
# Base = delivered messages
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. DPD buckets
# ------------------------------------------------------------

dpd_bins = [
    -np.inf,
    7,
    15,
    30,
    45,
    59,
    np.inf
]

dpd_labels = [
    "01-07",
    "08-15",
    "16-30",
    "31-45",
    "46-59",
    "60+"
]

eng["dpd_bucket"] = pd.cut(
    eng["days_past_due"],
    bins=dpd_bins,
    labels=dpd_labels
)

# ------------------------------------------------------------
# 2. Template × DPD × Weekday/Weekend
# ------------------------------------------------------------

dpd_template = (
    eng.groupby(
        ["template", "dpd_bucket", "send_day_type"],
        observed=True
    )
    .agg(
        messages=("message_id", "nunique"),
        customers=("customer_id", "nunique"),

        reads=("read_flag", "sum"),
        clicks=("click_flag", "sum"),
        replies=("reply_flag", "sum"),
        any_engagement=("any_engagement_flag", "sum"),
        no_engagement=("none_flag", "sum")
    )
    .reset_index()
)

for metric in [
    "reads",
    "clicks",
    "replies",
    "any_engagement",
    "no_engagement"
]:
    dpd_template[f"{metric}_rate"] = (
        dpd_template[metric] /
        dpd_template["messages"]
    )

display(
    dpd_template[
        [
            "template",
            "dpd_bucket",
            "send_day_type",
            "messages",
            "customers",
            "reads_rate",
            "clicks_rate",
            "replies_rate",
            "any_engagement_rate",
            "no_engagement_rate"
        ]
    ].style.format({
        "reads_rate": "{:.2%}",
        "clicks_rate": "{:.2%}",
        "replies_rate": "{:.2%}",
        "any_engagement_rate": "{:.2%}",
        "no_engagement_rate": "{:.2%}"
    })
)

,template,dpd_bucket,send_day_type,messages,customers,reads_rate,clicks_rate,replies_rate,any_engagement_rate,no_engagement_rate
0,discount_offer,31-45,Weekday,2422,1920,22.50%,17.96%,6.69%,47.15%,52.85%
1,discount_offer,31-45,Weekend,438,422,18.04%,15.98%,4.57%,38.58%,61.42%
2,discount_offer,46-59,Weekday,1468,1187,20.98%,17.37%,6.54%,44.89%,55.11%
3,discount_offer,46-59,Weekend,245,240,17.14%,10.20%,6.53%,33.88%,66.12%
4,discount_offer,60+,Weekday,85,85,10.59%,18.82%,2.35%,31.76%,68.24%
5,discount_offer,60+,Weekend,11,11,9.09%,18.18%,9.09%,36.36%,63.64%
6,friendly_reminder,01-07,Weekday,12930,8209,37.72%,9.13%,8.15%,55.00%,45.00%
7,friendly_reminder,01-07,Weekend,2106,2010,32.00%,6.46%,6.93%,45.39%,54.61%
8,friendly_reminder,08-15,Weekday,3036,2606,31.36%,7.25%,6.39%,44.99%,55.01%
9,friendly_reminder,08-15,Weekend,494,488,25.30%,4.05%,7.09%,36.44%,63.56%


In [13]:
# ============================================================
# WEEKEND OPTIMIZATION
# 04 — TEMPLATE × DPD × WEEKDAY/WEEKEND
# Base = delivered messages
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. DPD buckets
# ------------------------------------------------------------

dpd_bins = [
    -np.inf,
    7,
    15,
    30,
    45,
    59,
    np.inf
]

dpd_labels = [
    "01-07",
    "08-15",
    "16-30",
    "31-45",
    "46-59",
    "60+"
]

eng["dpd_bucket"] = pd.cut(
    eng["days_past_due"],
    bins=dpd_bins,
    labels=dpd_labels
)

# ------------------------------------------------------------
# 2. Template × DPD × Weekday/Weekend
# ------------------------------------------------------------

dpd_template = (
    eng.groupby(
        ["template", "dpd_bucket", "send_day_type"],
        observed=True
    )
    .agg(
        messages=("message_id", "nunique"),
        customers=("customer_id", "nunique"),

        reads=("read_flag", "sum"),
        clicks=("click_flag", "sum"),
        replies=("reply_flag", "sum"),
        any_engagement=("any_engagement_flag", "sum"),
        no_engagement=("none_flag", "sum")
    )
    .reset_index()
)

for metric in [
    "reads",
    "clicks",
    "replies",
    "any_engagement",
    "no_engagement"
]:
    dpd_template[f"{metric}_rate"] = (
        dpd_template[metric] /
        dpd_template["messages"]
    )

display(
    dpd_template[
        [
            "template",
            "dpd_bucket",
            "send_day_type",
            "messages",
            "customers",
            "reads_rate",
            "clicks_rate",
            "replies_rate",
            "any_engagement_rate",
            "no_engagement_rate"
        ]
    ].style.format({
        "reads_rate": "{:.2%}",
        "clicks_rate": "{:.2%}",
        "replies_rate": "{:.2%}",
        "any_engagement_rate": "{:.2%}",
        "no_engagement_rate": "{:.2%}"
    })
)

,template,dpd_bucket,send_day_type,messages,customers,reads_rate,clicks_rate,replies_rate,any_engagement_rate,no_engagement_rate
0,discount_offer,31-45,Weekday,2422,1920,22.50%,17.96%,6.69%,47.15%,52.85%
1,discount_offer,31-45,Weekend,438,422,18.04%,15.98%,4.57%,38.58%,61.42%
2,discount_offer,46-59,Weekday,1468,1187,20.98%,17.37%,6.54%,44.89%,55.11%
3,discount_offer,46-59,Weekend,245,240,17.14%,10.20%,6.53%,33.88%,66.12%
4,discount_offer,60+,Weekday,85,85,10.59%,18.82%,2.35%,31.76%,68.24%
5,discount_offer,60+,Weekend,11,11,9.09%,18.18%,9.09%,36.36%,63.64%
6,friendly_reminder,01-07,Weekday,12930,8209,37.72%,9.13%,8.15%,55.00%,45.00%
7,friendly_reminder,01-07,Weekend,2106,2010,32.00%,6.46%,6.93%,45.39%,54.61%
8,friendly_reminder,08-15,Weekday,3036,2606,31.36%,7.25%,6.39%,44.99%,55.01%
9,friendly_reminder,08-15,Weekend,494,488,25.30%,4.05%,7.09%,36.44%,63.56%


In [14]:
# ============================================================
# WEEKEND OPTIMIZATION
# 05 — CONTACT PRESSURE × WEEKEND
# Base = delivered messages
# ============================================================

pressure_bins = [-1, 1, 3, 5, np.inf]

pressure_labels = [
    "0-1 prior msgs",
    "2-3 prior msgs",
    "4-5 prior msgs",
    "6+ prior msgs"
]

eng["contact_pressure"] = pd.cut(
    eng["n_msgs_last_14d"],
    bins=pressure_bins,
    labels=pressure_labels
)

pressure = (
    eng.groupby(
        ["contact_pressure", "send_day_type"],
        observed=True
    )
    .agg(
        messages=("message_id", "nunique"),
        customers=("customer_id", "nunique"),

        reads=("read_flag", "sum"),
        clicks=("click_flag", "sum"),
        replies=("reply_flag", "sum"),
        any_engagement=("any_engagement_flag", "sum"),
        no_engagement=("none_flag", "sum")
    )
    .reset_index()
)

for metric in [
    "reads",
    "clicks",
    "replies",
    "any_engagement",
    "no_engagement"
]:
    pressure[f"{metric}_rate"] = (
        pressure[metric] /
        pressure["messages"]
    )

display(
    pressure.style.format({
        "reads_rate": "{:.2%}",
        "clicks_rate": "{:.2%}",
        "replies_rate": "{:.2%}",
        "any_engagement_rate": "{:.2%}",
        "no_engagement_rate": "{:.2%}"
    })
)

,contact_pressure,send_day_type,messages,customers,reads,clicks,replies,any_engagement,no_engagement,reads_rate,clicks_rate,replies_rate,any_engagement_rate,no_engagement_rate
0,0-1 prior msgs,Weekday,23180,10710,7377,3533,1919,12829,10351,31.82%,15.24%,8.28%,55.35%,44.65%
1,0-1 prior msgs,Weekend,3663,3260,965,444,247,1656,2007,26.34%,12.12%,6.74%,45.21%,54.79%
2,2-3 prior msgs,Weekday,20465,8266,5575,2597,1458,9630,10835,27.24%,12.69%,7.12%,47.06%,52.94%
3,2-3 prior msgs,Weekend,3404,2909,786,331,206,1323,2081,23.09%,9.72%,6.05%,38.87%,61.13%
4,4-5 prior msgs,Weekday,9007,4393,2103,949,543,3595,5412,23.35%,10.54%,6.03%,39.91%,60.09%
5,4-5 prior msgs,Weekend,1559,1377,266,119,85,470,1089,17.06%,7.63%,5.45%,30.15%,69.85%
6,6+ prior msgs,Weekday,1968,1068,394,154,87,635,1333,20.02%,7.83%,4.42%,32.27%,67.73%
7,6+ prior msgs,Weekend,297,268,39,18,10,67,230,13.13%,6.06%,3.37%,22.56%,77.44%


In [15]:
# ============================================================
# TEMPLATE × DPD × CONTACT PRESSURE
# WEEKEND ONLY
# ============================================================

weekend_pressure = (
    eng.loc[eng["send_day_type"].eq("Weekend")]
       .groupby(
           ["template", "dpd_bucket", "contact_pressure"],
           observed=True
       )
       .agg(
           messages=("message_id", "nunique"),
           customers=("customer_id", "nunique"),
           reads=("read_flag", "sum"),
           clicks=("click_flag", "sum"),
           replies=("reply_flag", "sum"),
           any_engagement=("any_engagement_flag", "sum"),
           no_engagement=("none_flag", "sum")
       )
       .reset_index()
)

for metric in [
    "reads",
    "clicks",
    "replies",
    "any_engagement",
    "no_engagement"
]:
    weekend_pressure[f"{metric}_rate"] = (
        weekend_pressure[metric] /
        weekend_pressure["messages"]
    )

# Ignore tiny cells for decision-making
decision_view = (
    weekend_pressure
    .loc[weekend_pressure["messages"] >= 50]
    .sort_values(
        ["no_engagement_rate", "messages"],
        ascending=[False, False]
    )
)

display(
    decision_view[
        [
            "template",
            "dpd_bucket",
            "contact_pressure",
            "messages",
            "customers",
            "reads_rate",
            "clicks_rate",
            "replies_rate",
            "any_engagement_rate",
            "no_engagement_rate"
        ]
    ].style.format({
        "reads_rate": "{:.2%}",
        "clicks_rate": "{:.2%}",
        "replies_rate": "{:.2%}",
        "any_engagement_rate": "{:.2%}",
        "no_engagement_rate": "{:.2%}"
    })
)

,template,dpd_bucket,contact_pressure,messages,customers,reads_rate,clicks_rate,replies_rate,any_engagement_rate,no_engagement_rate
48,urgent_reminder,16-30,6+ prior msgs,73,68,8.22%,4.11%,0.00%,12.33%,87.67%
27,pix_link,08-15,6+ prior msgs,50,50,6.00%,8.00%,4.00%,18.00%,82.00%
44,urgent_reminder,08-15,6+ prior msgs,77,77,14.29%,2.60%,5.19%,22.08%,77.92%
43,urgent_reminder,08-15,4-5 prior msgs,393,387,19.34%,1.27%,5.09%,25.70%,74.30%
54,urgent_reminder,46-59,2-3 prior msgs,50,49,18.00%,4.00%,4.00%,26.00%,74.00%
50,urgent_reminder,31-45,2-3 prior msgs,128,127,21.09%,3.91%,3.91%,28.91%,71.09%
47,urgent_reminder,16-30,4-5 prior msgs,322,314,20.81%,4.66%,3.73%,29.19%,70.81%
15,friendly_reminder,08-15,4-5 prior msgs,149,149,19.46%,4.03%,6.04%,29.53%,70.47%
30,pix_link,16-30,4-5 prior msgs,183,181,8.74%,16.39%,6.56%,31.69%,68.31%
5,discount_offer,46-59,2-3 prior msgs,89,88,15.73%,11.24%,5.62%,32.58%,67.42%


In [16]:
# ============================================================
# CANDIDATE SUPPRESSION POLICY #1
#
# Weekend
# + urgent_reminder
# + DPD 8-30
# + >= 4 prior messages in previous 14 days
# ============================================================

policy = eng.copy()

policy["suppress_candidate"] = (
    policy["send_day_type"].eq("Weekend")
    & policy["template"].eq("urgent_reminder")
    & policy["days_past_due"].between(8, 30)
    & policy["n_msgs_last_14d"].ge(4)
)

candidate = policy.loc[
    policy["suppress_candidate"]
].copy()

# ------------------------------------------------------------
# Volume
# ------------------------------------------------------------

total_delivered = len(policy)
candidate_messages = len(candidate)

volume_reduction_pct = (
    candidate_messages /
    total_delivered
)

# ------------------------------------------------------------
# Engagement
# ------------------------------------------------------------

candidate_engagement = candidate["any_engagement_flag"].sum()
candidate_none = candidate["none_flag"].sum()
candidate_clicks = candidate["click_flag"].sum()
candidate_reads = candidate["read_flag"].sum()
candidate_replies = candidate["reply_flag"].sum()

# ------------------------------------------------------------
# Share of ALL engagement potentially removed
# ------------------------------------------------------------

share_engagement = (
    candidate_engagement /
    policy["any_engagement_flag"].sum()
)

share_clicks = (
    candidate_clicks /
    policy["click_flag"].sum()
)

share_replies = (
    candidate_replies /
    policy["reply_flag"].sum()
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("=" * 75)
print("CANDIDATE SUPPRESSION POLICY #1")
print("=" * 75)

print("\nRULE")
print(
    "Weekend + urgent_reminder + DPD 8-30 "
    "+ >=4 prior messages"
)

print("\nVOLUME")
print(f"Messages removed                 : {candidate_messages:,}")
print(f"% delivered volume removed       : {volume_reduction_pct:.2%}")

print("\nCUSTOMER ATTENTION")
print(f"No-engagement messages removed    : {candidate_none:,}")
print(f"Engaged messages removed          : {candidate_engagement:,}")
print(f"Candidate engagement rate         : {candidate['any_engagement_flag'].mean():.2%}")
print(f"Candidate no-engagement rate      : {candidate['none_flag'].mean():.2%}")

print("\nENGAGEMENT AT RISK")
print(f"Reads potentially removed         : {candidate_reads:,}")
print(f"Clicks potentially removed        : {candidate_clicks:,}")
print(f"Replies potentially removed       : {candidate_replies:,}")

print("\nSHARE OF TOTAL ENGAGEMENT")
print(f"% total engagement affected       : {share_engagement:.2%}")
print(f"% total clicks affected           : {share_clicks:.2%}")
print(f"% total replies affected          : {share_replies:.2%}")

CANDIDATE SUPPRESSION POLICY #1

RULE
Weekend + urgent_reminder + DPD 8-30 + >=4 prior messages

VOLUME
Messages removed                 : 865
% delivered volume removed       : 1.36%

CUSTOMER ATTENTION
No-engagement messages removed    : 644
Engaged messages removed          : 221
Candidate engagement rate         : 25.55%
Candidate no-engagement rate      : 74.45%

ENGAGEMENT AT RISK
Reads potentially removed         : 160
Clicks potentially removed        : 25
Replies potentially removed       : 36

SHARE OF TOTAL ENGAGEMENT
% total engagement affected       : 0.73%
% total clicks affected           : 0.31%
% total replies affected          : 0.79%


In [17]:
# ============================================================
# PAYMENT / RECOVERY GUARDRAIL
# ============================================================

candidate_paid = candidate["paid_within_72h"].fillna(False).astype(bool)

all_paid = policy["paid_within_72h"].fillna(False).astype(bool)

candidate_payment_events = candidate_paid.sum()
all_payment_events = all_paid.sum()

candidate_recovery = candidate["amount_paid_brl"].fillna(0).sum()
all_recovery = policy["amount_paid_brl"].fillna(0).sum()

print("=" * 75)
print("PAYMENT / RECOVERY GUARDRAIL")
print("=" * 75)

print(f"Candidate payment events          : {candidate_payment_events:,}")
print(f"Candidate payment response        : {candidate_paid.mean():.2%}")
print()

print(
    f"% observed payment events affected: "
    f"{candidate_payment_events / all_payment_events:.2%}"
)

print(
    f"Attributed recovery               : "
    f"R$ {candidate_recovery:,.2f}"
)

print(
    f"% attributed recovery affected    : "
    f"{candidate_recovery / all_recovery:.2%}"
)

PAYMENT / RECOVERY GUARDRAIL
Candidate payment events          : 53
Candidate payment response        : 6.13%

% observed payment events affected: 0.94%
Attributed recovery               : R$ 34,124.13
% attributed recovery affected    : 0.99%


In [18]:
# ============================================================
# WEEKEND MESSAGE OPTIMIZATION
# SUPPRESSION SCENARIO CURVE
# ============================================================

base = eng.copy()

# Make payment flag robust
base["payment_flag"] = (
    base["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

# ------------------------------------------------------------
# Candidate scenarios
# ------------------------------------------------------------

scenarios = {

    "S1 | Weekend urgent | DPD 8-30 | 4+ prior": (
        base["send_day_type"].eq("Weekend")
        & base["template"].eq("urgent_reminder")
        & base["days_past_due"].between(8, 30)
        & base["n_msgs_last_14d"].ge(4)
    ),

    "S2 | Weekend urgent | DPD 8-30 | 2+ prior": (
        base["send_day_type"].eq("Weekend")
        & base["template"].eq("urgent_reminder")
        & base["days_past_due"].between(8, 30)
        & base["n_msgs_last_14d"].ge(2)
    ),

    "S3 | Weekend urgent | DPD 8-59": (
        base["send_day_type"].eq("Weekend")
        & base["template"].eq("urgent_reminder")
        & base["days_past_due"].between(8, 59)
    ),

    "S4 | Sunday urgent": (
        base["day_of_week"].eq("Sunday")
        & base["template"].eq("urgent_reminder")
    ),

    "S5 | Sunday urgent + friendly": (
        base["day_of_week"].eq("Sunday")
        & base["template"].isin(
            ["urgent_reminder", "friendly_reminder"]
        )
    )
}

# ------------------------------------------------------------
# Totals
# ------------------------------------------------------------

total_messages = len(base)
total_engagement = base["any_engagement_flag"].sum()
total_clicks = base["click_flag"].sum()
total_replies = base["reply_flag"].sum()
total_payments = base["payment_flag"].sum()
total_recovery = base["amount_paid_brl"].fillna(0).sum()

rows = []

# ------------------------------------------------------------
# Evaluate each scenario
# ------------------------------------------------------------

for scenario_name, mask in scenarios.items():

    x = base.loc[mask].copy()

    messages = len(x)

    engagement = x["any_engagement_flag"].sum()
    clicks = x["click_flag"].sum()
    replies = x["reply_flag"].sum()

    payments = x["payment_flag"].sum()

    recovery = (
        x["amount_paid_brl"]
        .fillna(0)
        .sum()
    )

    rows.append({

        "scenario": scenario_name,

        "messages_removed": messages,

        "volume_removed_pct":
            messages / total_messages * 100,

        "no_engagement_pct":
            x["none_flag"].mean() * 100,

        "engagement_rate_pct":
            x["any_engagement_flag"].mean() * 100,

        "total_engagement_affected_pct":
            engagement / total_engagement * 100,

        "total_clicks_affected_pct":
            clicks / total_clicks * 100,

        "total_replies_affected_pct":
            replies / total_replies * 100,

        "payment_response_pct":
            x["payment_flag"].mean() * 100,

        "total_payment_events_affected_pct":
            payments / total_payments * 100,

        "recovery_brl":
            recovery,

        "total_recovery_affected_pct":
            recovery / total_recovery * 100
    })

scenario_table = pd.DataFrame(rows)

display(
    scenario_table.style.format({

        "volume_removed_pct": "{:.2f}%",

        "no_engagement_pct": "{:.2f}%",
        "engagement_rate_pct": "{:.2f}%",

        "total_engagement_affected_pct": "{:.2f}%",
        "total_clicks_affected_pct": "{:.2f}%",
        "total_replies_affected_pct": "{:.2f}%",

        "payment_response_pct": "{:.2f}%",

        "total_payment_events_affected_pct": "{:.2f}%",

        "recovery_brl": "R$ {:,.2f}",

        "total_recovery_affected_pct": "{:.2f}%"
    })
)

,scenario,messages_removed,volume_removed_pct,no_engagement_pct,engagement_rate_pct,total_engagement_affected_pct,total_clicks_affected_pct,total_replies_affected_pct,payment_response_pct,total_payment_events_affected_pct,recovery_brl,total_recovery_affected_pct
0,S1 | Weekend urgent | DPD 8-30 | 4+ prior,865,1.36%,74.45%,25.55%,0.73%,0.31%,0.79%,6.13%,0.94%,"R$ 34,124.13",0.99%
1,S2 | Weekend urgent | DPD 8-30 | 2+ prior,1943,3.06%,67.68%,32.32%,2.08%,0.92%,2.06%,6.85%,2.36%,"R$ 79,513.42",2.30%
2,S3 | Weekend urgent | DPD 8-59,2660,4.19%,66.02%,33.98%,2.99%,1.25%,2.85%,6.69%,3.16%,"R$ 102,343.35",2.96%
3,S4 | Sunday urgent,1023,1.61%,70.67%,29.33%,0.99%,0.31%,0.94%,7.23%,1.32%,"R$ 46,455.96",1.34%
4,S5 | Sunday urgent + friendly,2172,3.42%,65.56%,34.44%,2.48%,1.12%,2.39%,8.20%,3.16%,"R$ 113,645.45",3.29%


In [19]:
# ============================================================
# INCREMENTAL SUPPRESSION ANALYSIS
# S1 -> S2 -> S3
# ============================================================

s1 = scenarios[
    "S1 | Weekend urgent | DPD 8-30 | 4+ prior"
]

s2 = scenarios[
    "S2 | Weekend urgent | DPD 8-30 | 2+ prior"
]

s3 = scenarios[
    "S3 | Weekend urgent | DPD 8-59"
]

incremental_scenarios = {
    
    # Core: highest-pressure segment
    "Core S1": s1,

    # What we ADD when moving S1 -> S2
    "Increment S1 -> S2": (
        s2 & ~s1
    ),

    # What we ADD when moving S2 -> S3
    "Increment S2 -> S3": (
        s3 & ~s2
    )
}

rows = []

for name, mask in incremental_scenarios.items():

    x = base.loc[mask].copy()

    messages = len(x)

    engagement = x["any_engagement_flag"].sum()
    clicks = x["click_flag"].sum()
    replies = x["reply_flag"].sum()

    payments = x["payment_flag"].sum()

    recovery = (
        x["amount_paid_brl"]
        .fillna(0)
        .sum()
    )

    rows.append({

        "layer": name,

        "messages": messages,

        "share_total_volume_pct":
            messages / total_messages * 100,

        "engagement_rate_pct":
            x["any_engagement_flag"].mean() * 100,

        "no_engagement_pct":
            x["none_flag"].mean() * 100,

        "click_rate_pct":
            x["click_flag"].mean() * 100,

        "reply_rate_pct":
            x["reply_flag"].mean() * 100,

        "payment_response_pct":
            x["payment_flag"].mean() * 100,

        "recovery_brl":
            recovery,

        "recovery_per_message":
            recovery / messages,

        "share_total_recovery_pct":
            recovery / total_recovery * 100,

        "share_total_payments_pct":
            payments / total_payments * 100
    })

incremental_table = pd.DataFrame(rows)

display(
    incremental_table.style.format({

        "share_total_volume_pct": "{:.2f}%",

        "engagement_rate_pct": "{:.2f}%",
        "no_engagement_pct": "{:.2f}%",

        "click_rate_pct": "{:.2f}%",
        "reply_rate_pct": "{:.2f}%",

        "payment_response_pct": "{:.2f}%",

        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",

        "share_total_recovery_pct": "{:.2f}%",
        "share_total_payments_pct": "{:.2f}%"
    })
)

,layer,messages,share_total_volume_pct,engagement_rate_pct,no_engagement_pct,click_rate_pct,reply_rate_pct,payment_response_pct,recovery_brl,recovery_per_message,share_total_recovery_pct,share_total_payments_pct
0,Core S1,865,1.36%,25.55%,74.45%,2.89%,4.16%,6.13%,"R$ 34,124.13",R$ 39.45,0.99%,0.94%
1,Increment S1 -> S2,1078,1.70%,37.76%,62.24%,4.64%,5.38%,7.42%,"R$ 45,389.29",R$ 42.11,1.31%,1.42%
2,Increment S2 -> S3,717,1.13%,38.49%,61.51%,3.77%,5.02%,6.28%,"R$ 22,829.93",R$ 31.84,0.66%,0.80%


In [21]:
# ============================================================
# DECOMPOSE S2 -> S3
# ============================================================

increment_low_pressure = (
    base["send_day_type"].eq("Weekend")
    & base["template"].eq("urgent_reminder")
    & base["days_past_due"].between(8, 30)
    & base["n_msgs_last_14d"].lt(2)
)

increment_high_dpd = (
    base["send_day_type"].eq("Weekend")
    & base["template"].eq("urgent_reminder")
    & base["days_past_due"].between(31, 59)
)

segments = {
    "DPD 8-30 | 0-1 prior": increment_low_pressure,
    "DPD 31-59 | all pressure": increment_high_dpd
}

rows = []

for name, mask in segments.items():

    x = base.loc[mask].copy()

    messages = len(x)
    payments = x["payment_flag"].sum()
    recovery = x["amount_paid_brl"].fillna(0).sum()

    rows.append({
        "segment": name,

        "messages": messages,

        "share_total_volume_pct":
            messages / total_messages * 100,

        "engagement_rate_pct":
            x["any_engagement_flag"].mean() * 100,

        "no_engagement_pct":
            x["none_flag"].mean() * 100,

        "click_rate_pct":
            x["click_flag"].mean() * 100,

        "reply_rate_pct":
            x["reply_flag"].mean() * 100,

        "payment_response_pct":
            x["payment_flag"].mean() * 100,

        "recovery_brl":
            recovery,

        "recovery_per_message":
            recovery / messages,

        "share_total_recovery_pct":
            recovery / total_recovery * 100,

        "share_total_payments_pct":
            payments / total_payments * 100
    })

decomposition = pd.DataFrame(rows)

display(
    decomposition.style.format({
        "share_total_volume_pct": "{:.2f}%",
        "engagement_rate_pct": "{:.2f}%",
        "no_engagement_pct": "{:.2f}%",
        "click_rate_pct": "{:.2f}%",
        "reply_rate_pct": "{:.2f}%",
        "payment_response_pct": "{:.2f}%",
        "recovery_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "share_total_recovery_pct": "{:.2f}%",
        "share_total_payments_pct": "{:.2f}%"
    })
)


,segment,messages,share_total_volume_pct,engagement_rate_pct,no_engagement_pct,click_rate_pct,reply_rate_pct,payment_response_pct,recovery_brl,recovery_per_message,share_total_recovery_pct,share_total_payments_pct
0,DPD 8-30 | 0-1 prior,334,0.53%,44.61%,55.39%,3.89%,5.39%,7.78%,"R$ 12,063.64",R$ 36.12,0.35%,0.46%
1,DPD 31-59 | all pressure,383,0.60%,33.16%,66.84%,3.66%,4.70%,4.96%,"R$ 10,766.29",R$ 28.11,0.31%,0.34%


In [22]:
# ============================================================
# TARGETED WEEKEND SUPPRESSION POLICY
# ============================================================

targeted_policy = (
    base["send_day_type"].eq("Weekend")
    & base["template"].eq("urgent_reminder")
    & (
        (
            base["days_past_due"].between(8, 30)
            & base["n_msgs_last_14d"].ge(4)
        )
        |
        base["days_past_due"].between(31, 59)
    )
)

x = base.loc[targeted_policy].copy()

messages = len(x)
customers = x["customer_id"].nunique()

engagement = x["any_engagement_flag"].sum()
clicks = x["click_flag"].sum()
replies = x["reply_flag"].sum()

payments = x["payment_flag"].sum()
recovery = x["amount_paid_brl"].fillna(0).sum()

targeted_summary = pd.DataFrame({
    "metric": [
        "Messages removed",
        "Customers affected",
        "Share total volume",
        "No engagement rate",
        "Engagement rate",
        "Click rate",
        "Reply rate",
        "Payment response",
        "Share total engagement",
        "Share total clicks",
        "Share total replies",
        "Share payment events",
        "Attributed recovery",
        "Share attributed recovery",
        "Recovery per message"
    ],
    "value": [
        f"{messages:,}",
        f"{customers:,}",
        f"{messages / total_messages:.2%}",
        f"{x['none_flag'].mean():.2%}",
        f"{x['any_engagement_flag'].mean():.2%}",
        f"{x['click_flag'].mean():.2%}",
        f"{x['reply_flag'].mean():.2%}",
        f"{x['payment_flag'].mean():.2%}",
        f"{engagement / total_engagement:.2%}",
        f"{clicks / total_clicks:.2%}",
        f"{replies / total_replies:.2%}",
        f"{payments / total_payments:.2%}",
        f"R$ {recovery:,.2f}",
        f"{recovery / total_recovery:.2%}",
        f"R$ {recovery / messages:,.2f}"
    ]
})

display(targeted_summary)

,metric,value
0,Messages removed,"1,248"
1,Customers affected,"1,135"
2,Share total volume,1.96%
3,No engagement rate,72.12%
4,Engagement rate,27.88%
5,Click rate,3.12%
6,Reply rate,4.33%
7,Payment response,5.77%
8,Share total engagement,1.15%
9,Share total clicks,0.48%
